<a href="https://colab.research.google.com/github/cujoramirez/StyleTailor/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the data into DataFrames and set column names.
names = ["class", "message"]

train_file = pd.read_csv(train_file_path, sep='\t', names=names)
test_file = pd.read_csv(test_file_path, sep='\t', names=names)

print(train_file.head())
print(test_file.head())


In [ ]:
# Preprocess the data:

# Convert class labels to numeric.
train_labels = np.array([0 if x=="ham" else 1 for x in train_file['class'].values])
test_labels = np.array([0 if x=="ham" else 1 for x in test_file['class'].values])

# Set parameters for text vectorization.
max_features = 10000
sequence_length = 50

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.adapt(train_file['message'].values)

x_train = vectorize_layer(train_file['message'].values)
x_test = vectorize_layer(test_file['message'].values)

model = keras.Sequential([
    keras.Input(shape=(sequence_length,)),
    keras.layers.Embedding(max_features, 64),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Use EarlyStopping to prevent over-training.
early_stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=1e-4, patience=25, restore_best_weights=True, verbose=1)

history = model.fit(x_train, train_labels,
                    validation_data=(x_test, test_labels),
                    epochs=1000,
                    callbacks=[early_stop],
                    verbose=2)


In [ ]:
# function to predict messages based on model
def predict_message(pred_text):
    class_dict = {0: "ham", 1: "spam"}
    vect_text = vectorize_layer([pred_text])
    prob = model.predict(vect_text)[0][0]
    label = class_dict[int(np.round(prob))]
    return [prob, label]

# Test a sample message.
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
